# Caching in Nodes
Companion notebook for the **Caching in Nodes** section of Chapter 4 (Cache Policy and Cache Store approaches).

LangGraph can skip re-running a node when it is called again with the exact same input, as long as the node was registered with a `cache_policy` and the graph was compiled with a `cache` store.

In [1]:
import time
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import CachePolicy
from langgraph.cache.memory import InMemoryCache


class ResearchState(BaseModel):
    query: str = ""
    candidate_urls: list[str] = Field(default_factory=list)

`search_sources` below stands in for an expensive operation (a real web search or LLM call). A module-level counter lets us prove, from the outside, whether it actually ran.

In [2]:
call_count = {"search_sources": 0}


def search_sources(state: ResearchState) -> dict:
    """Pretend to search the web. cache_policy lets LangGraph skip this on identical repeated input."""
    call_count["search_sources"] += 1
    time.sleep(0.05)  # simulate an expensive I/O call
    return {"candidate_urls": [f"https://example.com/{state.query}"]}

**Cache Policy approach** -- attach a `cache_policy` to the node when registering it:

In [3]:
builder = StateGraph(ResearchState)
builder.add_node(
    "search_sources",
    search_sources,
    cache_policy=CachePolicy(ttl=120),  # cache results 2 minutes
)
builder.add_edge(START, "search_sources")
builder.add_edge("search_sources", END)

**Cache Store approach** -- pass a cache backend when compiling the graph:

In [4]:
graph = builder.compile(cache=InMemoryCache())

In [5]:
graph.invoke({"query": "battery technology"})
print("Calls after 1st invoke:", call_count["search_sources"])

graph.invoke({"query": "battery technology"})
print("Calls after 2nd invoke (same query, should be cached):", call_count["search_sources"])

graph.invoke({"query": "quantum computing"})
print("Calls after 3rd invoke (different query, should run again):", call_count["search_sources"])

Calls after 1st invoke: 1
Calls after 2nd invoke (same query, should be cached): 1
Calls after 3rd invoke (different query, should run again): 2
